In [1]:
from cellgrn.main import normalzie_rna,parse_edges,compute_all_cells_grn,summarize_grn,format_sample_grn,format_celltype_grn,  SparseGRNCalculator, normalzie_rna_sparse

import numpy as np
import pandas as pd
import os
import anndata as ad
from scipy import sparse
import time
import gc
import tracemalloc

In [2]:
# n_cell = "c500"
soft = "linger"



cand_df = pd.read_csv(f"/home/shaliu_fu/multireg/cellGRN/data/bmmc/{soft}_grn.csv",header=0)

input_genes = [i.rstrip() for i in open(f"/home/shaliu_fu/multireg/cellGRN/data/bmmc//{soft}_gene.txt")]
input_peaks = [i.rstrip() for i in open(f"/home/shaliu_fu/multireg/cellGRN/data/bmmc//{soft}_peak.txt")]
all_tf = [i.rstrip() for i in open("/home/shaliu_fu/multireg/multigrn/db/all_hg_TF.txt")] 

In [ ]:
all_stats = []

for n_cell in ["c500", "c1k", "c5k", "c10k", "c50k"]:

    print(f"Processing {n_cell}...")
    
    # --- 开始监控 ---
    start_time = time.time()
    tracemalloc.start() # 启动内存追踪
    
    try:
        base_path = f"/home/shaliu_fu/multireg/benchmark/bench_dataset/scalability/{n_cell}"
        
        # 1. 加载数据
        cell_meta = pd.read_csv(f"{base_path}/metadata.csv", index_col=0)
        cell_types = cell_meta['cell_type.l1'].values

        input_rna = ad.read_h5ad(f"{base_path}/BMMC-multiome-{n_cell}-RNA-counts.h5ad")
        input_atac = ad.read_h5ad(f"{base_path}/BMMC-multiome-{n_cell}-ATAC-peaks.h5ad")
        
        input_gene = input_rna.var.index.values
        input_peak = input_atac.var.index.values
        input_tf = list(set(input_gene) & set(all_tf))
        cell_types = cell_meta['cell_type.l1']


        input_df1 = pd.DataFrame(input_rna.X.toarray(),index=input_rna.obs.index.values,columns=input_rna.var.index.values)
        peak_rename = [i.replace("-",":",1) for i in input_atac.var.index.values]
        input_df2 = pd.DataFrame(input_atac.X.toarray(),index=input_atac.obs.index.values,columns=peak_rename)

        # input_peaks = [i.replace(":","-") for i in input_peaks]
        input_df1 = input_df1[input_genes]
        input_df2 = input_df2[input_peaks]


        rna_data1,rna_data2 = normalzie_rna(input_df1)
        atac_data = input_df2.copy()

        input_tfs = [tf for tf in input_tf if tf in input_genes]
        tf_data1 = rna_data1[input_tfs].copy()
        tf_data2 = rna_data2[input_tfs].copy()

        edges_idx,edges_name = parse_edges(cand_df, input_tfs, input_genes, input_peaks)


        grn_scale2 = compute_all_cells_grn(tf_data2, rna_data2, atac_data,edges_idx, edges_name,
            input_tfs, input_genes, input_peaks)

        sample_grn_scale2, celltype_grn_scale2 = summarize_grn(grn_scale2, cell_types)


        tf_gene_res_scale2, tf_peak_res_scale2, gene_peak_res_scale2 = format_sample_grn(sample_grn_scale2)
        tf_gene_ct_res_scale2, tf_peak_ct_res_scale2, gene_peak_ct_res_scale2 = format_celltype_grn(celltype_grn_scale2)
        
    
    finally:
        # --- 结束监控 ---
        current, peak = tracemalloc.get_traced_memory() # 获取当前和峰值内存
        tracemalloc.stop() # 停止追踪
        end_time = time.time()
        
        run_time = round(end_time - start_time, 2)
        peak_mb = round(peak / (1024 * 1024), 2) # 转换为 MB
        
        print(f"[{n_cell}] Time: {run_time}s, Peak Memory: {peak_mb} MB")
        
        all_stats.append({
            "dataset": n_cell,
            "time_sec": run_time,
            "peak_memory_mb": peak_mb
        })

        # 清理内存
        if 'input_rna' in locals(): del input_rna
        if 'input_atac' in locals(): del input_atac
        if 'grn_scale2' in locals(): del grn_scale2
        gc.collect()

Processing c500...
[c500] Time: 139.4s, Peak Memory: 4399.55 MB
Processing c1k...
[c1k] Time: 220.41s, Peak Memory: 8695.05 MB
Processing c5k...
[c5k] Time: 421.61s, Peak Memory: 43059.33 MB
Processing c10k...
[c10k] Time: 704.98s, Peak Memory: 86018.12 MB
Processing c50k...


In [ ]:
df_stats = pd.DataFrame(all_stats)
df_stats.to_csv("/home/shaliu_fu/multireg/cellGRN/eval/results/scalability2.csv",header=True,index=None)

In [ ]:
# all_time = np.array(all_time)
# out = pd.DataFrame({"cell_num":[500,1000,5000,100000,50000],"time":all_time})
# out.to_csv("/home/shaliu_fu/multireg/cellGRN/eval/results/scalability.csv",header=True,index=None)

In [ ]:
all_stats = []

for n_cell in ["c500", "c1k", "c5k", "c10k", "c50k"]:

    print(f"Processing {n_cell}...")
    
    # --- 开始监控 ---
    start_time = time.time()
    tracemalloc.start() # 启动内存追踪
    
    try:
        base_path = f"/home/shaliu_fu/multireg/benchmark/bench_dataset/scalability/{n_cell}"
        
        # 1. 加载数据
        cell_meta = pd.read_csv(f"{base_path}/metadata.csv", index_col=0)
        cell_types = cell_meta['cell_type.l1'].values

        input_rna = ad.read_h5ad(f"{base_path}/BMMC-multiome-{n_cell}-RNA-counts.h5ad")
        input_atac = ad.read_h5ad(f"{base_path}/BMMC-multiome-{n_cell}-ATAC-peaks.h5ad")
        
        input_gene_names = input_rna.var.index.values
        peak_rename = [i.replace("-",":",1) for i in input_atac.var.index.values]
        input_atac.var.index = peak_rename
        input_peak_names = input_atac.var.index.values


        gene_indices = [input_rna.var_names.get_loc(g) for g in input_genes if g in input_rna.var_names]
        peak_indices = [input_atac.var_names.get_loc(p) for p in input_peaks if p in input_atac.var_names]
        
        rna_sub = input_rna[:, gene_indices].copy()
        atac_sub = input_atac[:, peak_indices].copy()
        
        current_genes = rna_sub.var.index.values
        current_peaks = atac_sub.var.index.values
        
        input_tfs = [tf for tf in current_genes if tf in all_tf]
        tf_indices_in_sub = [rna_sub.var_names.get_loc(tf) for tf in input_tfs]

        # 2. 归一化 (使用 MaxAbsScaler 修复版)
        rna_data1_sparse, rna_data2_dense = normalzie_rna_sparse(rna_sub.X)
        tf_data2_arr = rna_data2_dense[:, tf_indices_in_sub]

        # 3. 解析边与初始化计算器
        edges_idx, edges_name = parse_edges(cand_df, input_tfs, current_genes, current_peaks)
        calculator = SparseGRNCalculator(
            edges_idx, edges_name,
            input_tfs, current_genes, current_peaks
        )

        # 4. 分块计算
        batch_size = 1000 
        n_total = rna_sub.shape[0]
        atac_sparse = atac_sub.X 

        for i in range(0, n_total, batch_size):
            end_i = min(i + batch_size, n_total)
            
            batch_tf = tf_data2_arr[i:end_i]
            batch_rna = rna_data2_dense[i:end_i]
            batch_atac = atac_sparse[i:end_i]
            batch_ct = cell_types[i:end_i]
            
            calculator.process_batch(batch_tf, batch_rna, batch_atac, batch_ct)
            
            if i % 5000 == 0:
                gc.collect() # 显式回收垃圾

        # 5. 汇总结果
        sample_grn_scale2, celltype_grn_scale2 = calculator.finalize()
        
        # 格式化输出 (略)
        # tf_gene_res_scale2, ... = format_sample_grn(...)

    except Exception as e:
        print(f"Error in {n_cell}: {e}")
    
    finally:
        # --- 结束监控 ---
        current, peak = tracemalloc.get_traced_memory() # 获取当前和峰值内存
        tracemalloc.stop() # 停止追踪
        end_time = time.time()
        
        run_time = round(end_time - start_time, 2)
        peak_mb = round(peak / (1024 * 1024), 2) # 转换为 MB
        
        print(f"[{n_cell}] Time: {run_time}s, Peak Memory: {peak_mb} MB")
        
        all_stats.append({
            "dataset": n_cell,
            "time_sec": run_time,
            "peak_memory_mb": peak_mb
        })

        # 清理内存
        if 'input_rna' in locals(): del input_rna
        if 'input_atac' in locals(): del input_atac
        if 'rna_sub' in locals(): del rna_sub
        if 'atac_sub' in locals(): del atac_sub
        if 'rna_data2_dense' in locals(): del rna_data2_dense
        if 'calculator' in locals(): del calculator
        gc.collect()

# 打印最终统计表
print("\n=== Final Statistics ===")
df_stats = pd.DataFrame(all_stats)
print(df_stats)
df_stats.to_csv("/home/shaliu_fu/multireg/cellGRN/eval/results/scalability3.csv",header=True,index=None)